# 20 - Analogo en R (recreado desde cero)

## Tema base

**20 - Sesion especial: simulacion en programacion, de variable aleatoria a SIR con POO**


## Objetivos de la sesion en R

1. Mantener teoria del notebook Python en equivalente R.
2. Usar codigo 100% R ejecutable.
3. Conservar ejercicios adaptados a R.
4. Integrar extra de probabilidad y estadistica.


## Ruta Teoria-Codigo (alternada)

Se organiza la sesion en bloques cortos que alternan concepto y practica.


### Bloque teorico 1

### Teoria 1

# 20 - Sesion especial: simulacion en programacion, de variable aleatoria a SIR con POO

## Objetivos de aprendizaje

En esta sesion aprenderas a:

1. Entender que significa simular un sistema en programacion.
2. Conectar el concepto de variable aleatoria con codigo ejecutable.
3. Construir distribuciones empiricas a partir de muchas repeticiones.
4. Distinguir entre resultado de una corrida y comportamiento promedio.
5. Modelar un proceso dinamico paso a paso.
6. Implementar un modelo SIR estocastico.
7. Encapsular la simulacion del SIR con Programacion Orientada a Objetos.
8. Detectar errores comunes de modelado, interpretacion y programacion.

### Teoria 2

## 1. Que significa simular en programacion

Simular no es "adivinar" ni "dibujar datos".
Simular es:

- definir reglas explicitas de un sistema,
- ejecutar esas reglas muchas veces,
- observar resultados posibles bajo incertidumbre.

Cuando el sistema tiene azar, una sola corrida no representa toda la historia.
Por eso se repite, se resume y se interpreta.

### Teoria 3

## 2. Azar computacional: pseudoaleatorio y semilla

En computadora usamos generadores pseudoaleatorios.
Con la misma semilla (`seed`) obtenemos la misma secuencia, util para depurar y comparar.

### Teoria 4

## 3. Variable aleatoria: primer modelo con Bernoulli

Una variable aleatoria asigna un numero a cada resultado posible de un experimento.
Ejemplo Bernoulli:

- `1` si ocurre evento (contagio, compra, exito),
- `0` si no ocurre.

Eso ya permite medir probabilidades, promedios y riesgos.

### Teoria 5

## 4. Distribucion empirica: repetir para entender comportamiento

Una corrida puede salir rara.
Muchas corridas revelan patrones estables.


In [ ]:
set.seed(2026)
bernoulli <- function(p){if(p<0||p>1) stop('p en [0,1]'); as.integer(runif(1)<p)}
replicate(10, bernoulli(0.3))


### Bloque teorico 2

### Teoria 6

## 5. Monte Carlo: estimar probabilidades por simulacion

Problema: ?cual es la probabilidad de tener al menos 2 fallas en 8 componentes,
si cada componente falla con probabilidad 0.15 de forma independiente?

Puedes resolverlo con formula cerrada o con simulacion.
La simulacion es clave cuando la formula exacta no es sencilla.

### Teoria 7

## 6. De experimento aislado a sistema dinamico

Un sistema dinamico tiene estado que cambia en el tiempo.
En epidemiologia, ese estado puede ser cuantas personas estan:

- Susceptibles (`S`),
- Infectadas (`I`),
- Recuperadas (`R`).

Eso es el nucleo del modelo SIR.

### Teoria 8

## 7. Modelo SIR estocastico (version procedural minima)

Primero lo implementamos de forma procedural para entender reglas.
Luego lo pasamos a POO para mejorar claridad y mantenibilidad.

### Teoria 9

## 8. Limitacion de la version procedural

La version procedural funciona, pero mezcla:

- validaciones,
- estado,
- reglas de transicion,
- registro de historial.

Con POO podemos encapsular todo eso en una sola abstraccion coherente.

### Teoria 10

## 9. Aplicacion final: modelo SIR con Programacion Orientada a Objetos


In [ ]:
sim_evento <- function(){fallas <- sum(replicate(8, bernoulli(0.15))); fallas >= 2}
set.seed(7)
p_est <- mean(replicate(20000, sim_evento()))
p_est


### Bloque teorico 3

### Teoria 11

## 10. Comparar escenarios (pensar en decisiones, no solo ejecutar)

Cambiar `beta` y `gamma` cambia la dinamica.
La pregunta util no es "que salida me dio", sino:

- ?por que ese patron?
- ?que supuesto del modelo lo explica?
- ?que decision cambiaria el resultado?

### Teoria 12

## 11. Visualizacion rapida (opcional con matplotlib)

Si `matplotlib` esta disponible, puedes graficar.
Si no, el codigo sigue funcionando y puedes inspeccionar tablas.

### Teoria 13

## 12. Errores comunes al simular (remarcados)

1. Confundir una corrida con una conclusion general.
2. No fijar semilla cuando quieres depurar o comparar cambios.
3. Fijar semilla *dentro* de cada iteracion (rompe la variabilidad).
4. No validar parametros (probabilidades fuera de `[0, 1]`, poblacion negativa).
5. Olvidar invariantes del sistema (`S + I + R` debe mantenerse constante).
6. Mezclar unidades de tiempo sin declararlo (dias vs semanas).
7. Ajustar parametros hasta que "se vea bien" sin justificar supuestos.
8. Usar simulacion cuando hay solucion exacta simple y no explicarla.
9. Ignorar incertidumbre: reportar solo promedio sin dispersion.
10. Crear clases que solo almacenan datos sin encapsular reglas de negocio.

### Teoria 14

## 14. Resumen de conceptos clave

1. Simular es ejecutar reglas explicitas bajo incertidumbre.
2. Una variable aleatoria es una forma formal de representar resultados numericos de un experimento.
3. Monte Carlo aproxima probabilidades cuando repetir es mas simple que derivar formula.
4. En sistemas dinamicos importa la evolucion temporal del estado, no solo un valor final.
5. El modelo SIR separa poblacion en `S`, `I` y `R` para estudiar contagio y recuperacion.
6. POO ayuda a encapsular estado, invariantes y reglas de transicion.
7. Las conclusiones deben considerar variabilidad entre corridas, no solo una salida.


In [ ]:
simular_sir <- function(S0=95,I0=5,R0=0,beta=0.25,gamma=0.10,pasos=80){S <- integer(pasos+1); I <- integer(pasos+1); R <- integer(pasos+1); S[1]<-S0; I[1]<-I0; R[1]<-R0; N<-S0+I0+R0; for(t in 1:pasos){p_inf <- min(1,beta*I[t]/N); nuevos_inf <- rbinom(1,S[t],p_inf); nuevos_rec <- rbinom(1,I[t],gamma); S[t+1] <- S[t]-nuevos_inf; I[t+1] <- I[t]+nuevos_inf-nuevos_rec; R[t+1] <- R[t]+nuevos_rec}; data.frame(t=0:pasos,S=S,I=I,R=R)}
set.seed(42); tray <- simular_sir(); head(tray)


## Extra de probabilidad y estadistica

Monte Carlo, distribuciones empiricas y variabilidad de trayectorias SIR.


In [ ]:
matplot(tray$t, tray[,c('S','I','R')], type='l', lwd=2, col=c('steelblue','firebrick','darkgreen'))
legend('right', legend=c('S','I','R'), col=c('steelblue','firebrick','darkgreen'), lty=1, lwd=2)


## Analogias R y Python

- Simulacion estocastica en R y Python comparte la misma logica de reglas + repeticiones.


In [ ]:
# Checklist rapido R vs Python
# 1) ?Que cambia en sintaxis?
# 2) ?Que cambia en estructura de datos?
# 3) ?Que cambia en manejo de NA/errores?


## Errores comunes

- Concluir con una sola corrida.
- Ignorar semilla.
- No validar parametros probabilisticos.


In [ ]:
# Auto-verificacion de errores comunes
# Define un caso borde y valida que tu solucion en R no falle silenciosamente.


## Ejercicios para pensar (no copiar codigo)

Primero argumenta en texto. Luego, si hace falta, valida con experimentos en R.


### Ejercicio reflexivo 1

### Ejercicio 1

## Ruta de la sesion (secuencia ideal)

1. Que es simular.
2. Azar computacional y semillas.
3. Variable aleatoria (Bernoulli y conteos).
4. Estimacion por Monte Carlo.
5. Del experimento aislado al sistema con estado.
6. Modelo SIR estocastico (primero procedural).
7. Refactor a POO con una clase `SIRSimulador`.
8. Analisis de escenarios.
9. Errores comunes.
10. Ejercicios de pensamiento y diseno.


### Ejercicio reflexivo 2

### Ejercicio 2

## 13. Ejercicios de pensamiento y practica real

Estos ejercicios buscan razonamiento de modelado.
No basta con que el codigo "corra": debes justificar supuestos y decisiones.


### Ejercicio reflexivo 3

### Ejercicio 3

### Ejercicio 1: variable aleatoria bien definida

Disena una variable aleatoria `X` para "retraso de autobus" en minutos.

1. Define claramente el experimento aleatorio.
2. Propone dos distribuciones distintas para `X` y justifica cual usar.
3. Explica como cambiaria `E[X]` si hay hora pico.


### Ejercicio reflexivo 4

### Ejercicio 4

### Ejercicio 2: tamano de muestra y confianza

Quieres estimar por simulacion la probabilidad de que `I >= 40` antes del dia 50.

1. ?Cuantas corridas haras y por que?
2. ?Que metrica de estabilidad usaras para decidir si ya es suficiente?
3. Escribe pseudocodigo antes de programar.


### Ejercicio reflexivo 5

### Ejercicio 5

### Ejercicio 3: detectar un bug de aleatoriedad

Analiza el siguiente codigo y explica por que esta mal.
Luego corrigelo.


### Ejercicio reflexivo 6

### Ejercicio 6

### Ejercicio 4: invariantes del modelo SIR

Antes de tocar codigo, escribe al menos 4 invariantes de `SIRSimulador`.
Ejemplos esperados: no negatividad, conservacion de poblacion, etc.

Luego agrega `assert` o validaciones para hacerlas ejecutables.


### Ejercicio reflexivo 7

### Ejercicio 7

### Ejercicio 5: extender el modelo con vacunacion

Agrega una regla: cada paso se vacunan `v` personas susceptibles y pasan a `R`.

1. Decide donde vive esa regla (dentro de la clase o como estrategia externa).
2. Implementa la extension sin romper el API actual.
3. Explica tradeoffs de tu diseno.


### Ejercicio reflexivo 8

### Ejercicio 8

### Ejercicio 6: separar politicas de contagio y recuperacion

Refactoriza para desacoplar reglas de contagio/recuperacion del simulador.
Pista: usa objetos estrategia o funciones inyectadas.

Objetivo: probar facil distintos supuestos sin reescribir la clase principal.


### Ejercicio reflexivo 9

### Ejercicio 9

### Ejercicio 7: interpretar, no solo graficar

Corre 30 simulaciones con mismos parametros y semillas distintas.

1. Reporta promedio y desviacion estandar del pico de infectados.
2. Explica que significa alta dispersion para una decision publica.
3. ?Que recomendarias a una autoridad si el promedio se ve "aceptable" pero la cola de riesgo es alta?


### Ejercicio reflexivo 10

### Ejercicio 10

### Ejercicio 8: limites del modelo

Escribe una critica tecnica del SIR usado aqui.

Incluye:
1. dos supuestos poco realistas,
2. una variable relevante que falta,
3. una mejora concreta y como la modelarias.


### Ejercicio reflexivo 11

### Ejercicio 11

## 15. Reto final opcional

Construye una mini investigacion reproducible con tu `SIRSimulador`:

1. Define una pregunta de politica publica (por ejemplo: "?cuanto reducir `beta` para bajar el pico bajo 20?").
2. Corre al menos 50 simulaciones por escenario.
3. Compara tres estrategias (sin intervencion, reduccion de contacto, aumento de recuperacion).
4. Presenta resultados con una conclusion argumentada y limites del analisis.


### Preguntas de cierre

1. ?Que supuesto estadistico es mas fragil en esta clase?
2. ?Que evidencia minima pedirias antes de usar este enfoque en un caso real?
3. ?Como explicarias este tema a alguien no tecnico sin perder rigor?
4. ?Que cambiaria entre una implementacion en R y una en Python para produccion?
